## Data preprocessing

This notebook is for developing a data preprocessing pipeline for the ASD aggression dataset.

In [ ]:
from sys import path
import pandas as pd
import numpy as np
from copy import deepcopy

from physioview.pipeline import PPG, EDA
from physioview.pipeline import SQA

path.append('../shared/')
import utils
import viz_raw_and_processed as viz
import physio_processing as pp

%load_ext autoreload
%autoreload 2
%matplotlib inline

In [2]:
raw_data_base_path = '../../CBS_DATA_ASD_ONLY'
# List all session files
all_sessions, participants = utils.get_all_session_files(raw_data_base_path)
all_sessions_combined = [item for item in all_sessions if 'combined' in item]
print(f"Total number of participants: {len(set(participants))}")
print(f"Total number of sessions: {len(all_sessions_combined)}")

Total number of participants: 70
Total number of sessions: 426


### Preprocess BVP files

In [ ]:
# selects patient & session
t = 't1'
patient_id = '4356.01'
session = '04'
bvp_fs = 64

In [34]:
df_bvp = viz.get_df_raw_data_modality(raw_data_base_path, t, patient_id,session, 'BVP')
print(f"Sampling rate: {len(df_bvp) / (df_bvp['Timestamp'].iloc[-1] - df_bvp['Timestamp'].iloc[0]).total_seconds()}")

Sampling rate: 64.00020529665366


In [35]:
df_bvp['Timestamp'] = pd.to_datetime(df_bvp['Timestamp'], unit='ms')

filt = PPG.Filters(fs=bvp_fs)
filtered = filt.filter_signal(df_bvp['BVP'], lowcut=0.5, highcut=4.0, order=4, window_len=0.5)

df_bvp['Filtered'] = filtered

beat_detector = PPG.BeatDetectors(fs=bvp_fs, preprocessed=True)
peaks = beat_detector.adaptive_threshold(df_bvp['Filtered'])
df_bvp.loc[peaks, 'Peak'] = 1

cardio = SQA.Cardio(fs=bvp_fs)
beats_ix = df_bvp.loc[df_bvp['Peak'] == 1].index
artifacts_ix = cardio.identify_artifacts(beats_ix=beats_ix, method='both')
df_bvp['Artifact'] = np.nan
df_bvp.loc[artifacts_ix, 'Artifact'] = 1

# Get instantaneous heart rate
hr, peak_idx_for_hr = pp.get_instantaneous_heart_rate(pd.Series(df_bvp.loc[df_bvp['Peak'] == 1].index), fs=bvp_fs)
artifactual_hr_i = [i for i, idx in enumerate(peak_idx_for_hr) if idx in artifacts_ix]
hr_filtered = deepcopy(hr)
hr_filtered[artifactual_hr_i] = np.nan

# interpolate nan values
hr_interpolated = np.interp(np.where(np.isnan(hr_filtered))[0], np.where(~np.isnan(hr_filtered))[0], hr_filtered[~np.isnan(hr_filtered)])
hr_filtered[np.where(np.isnan(hr_filtered))[0]] = hr_interpolated
# Smooth the HR signal
hr_smoothed = pp.moving_average(hr_filtered, 20)
hr_smoothed = np.insert(hr_smoothed, 0, np.nan)
# Combine with the original data frame
df_bvp.loc[df_bvp['Peak'] == 1, 'HR'] = [h for h in hr_smoothed]

# Get RMSSD
window_size = 60
step_size = 15
rmssd, rmssd_timestamps = pp.get_rmssd(beats_ix, bvp_fs, window_size, step_size, artifacts_ix=artifacts_ix, beat_timestamps=df_bvp.loc[beats_ix, 'Timestamp'])
# interpolate nan values
rmssd_interpolated = np.interp(np.where(np.isnan(rmssd))[0], np.where(~np.isnan(rmssd))[0], rmssd[~np.isnan(rmssd)])
rmssd[np.where(np.isnan(rmssd))[0]] = rmssd_interpolated
# Smooth the RMSSD signal
rmssd_smoothed = pp.moving_average(rmssd, 3)
# Create RMSSD dataframe
df_rmssd = pd.DataFrame({'Timestamp': rmssd_timestamps, 'RMSSD': rmssd_smoothed})

/Users/yuna.w/Research/CBSL/aggression/ASD_agg_deep_learning/notebooks/../shared/physio_processing.py:56: RuntimeWarning:

Mean of empty slice



In [ ]:
# Visualize filtered signal and detected peaks
window_size = 60
modalities = ['Filtered']
viz.interactive_plot(df_bvp, window_size, modalities, show_ppg_peaks=True, show_artifacts=True)

interactive(children=(IntSlider(value=0, description='start_window', max=82), Output()), _dom_classes=('widget…

In [37]:
# Visualize the instantaneous heart rate
viz.plot_instantaneous_heart_rate(df_bvp)

In [38]:
# Plot RMSSD along with aggression events
viz.plot_rmssd(df_rmssd, df_bvp)